# GLiNER2 보안 감사 온톨로지 SFT v2: 관리대장과 서술 문서

가상 군부대 보안 감사 문서의 엔티티·관계·기록 레코드를 하나의 LoRA 어댑터로 학습한다. 목표는 관리대장 열 표현과 이름이 달라져도 같은 의미의 필드를 추출하는지 확인하는 것이다. 이상 여부 판정과 PDF/HWP 변환은 범위 밖이다.

v1 시험은 커밋 `9f18cde`와 `results/security_audit_sft_pilot.json`에 보존한다. 이 노트북은 **2026-11-13 로컬 NVIDIA GB10 v2 실행** 결과를 정리한다. 외부 API는 호출하지 않았으며 아래 셀은 저장 결과를 읽기만 한다.

## 결과 요약

| 평가 대상 | zero-shot | 통합 LoRA | 목표 |
|---|---|---|---|
| 엔티티 F1 | 0.444 | 0.979 | 0.90 |
| 관계 F1 | 0.050 | 0.772 | 0.80 |
| 일반 레코드 필드 F1 | 0.594 | 0.965 | 0.80 |
| 선택형 정확도 | 0.614 | 0.493 | 0.90 |

선택형 정확도는 레코드 필드 F1에서 제외하고 별도 집계한다. 위 점수는 최종 eval이며 dev는 아래에 따로 표시한다. 목표는 설계 시 정한 참고 기준으로, 이를 넘었다고 실문서 적용 성능이 보장되지는 않는다.

선택형 전체 정확도는 **0.614 → 0.493(175/355)**으로 하락했다. 엔티티와 일반 필드의 높은 F1이 선택형 실패를 가리지 않도록 별도 해석해야 한다. 관계 F1 0.772도 목표 0.80에 미달했다.


## 실행과 결과 파일

프로젝트 루트에서 다음 명령으로 실행한다. 기본 비교는 미병합 LoRA이며 CPU로 자동 전환하지 않는다.

```bash
uv run python scripts/security_audit/run_sft.py --device cuda --bf16 --cpu-threads 4 \
  --run-dir outputs/security_audit_gb10_v2 --stage all
```

실행 로그·예측·manifest·자원 기록은 `outputs/security_audit_gb10_v2/`에 있고, 검증 완료 결과를 `experiments/results/security_audit_sft.json`에 반영했다. 평가 중단 시 같은 설정·코드에서 `--stage evaluate --resume`으로 예측 체크포인트를 재사용한다. 학습 optimizer 상태의 중간 복원은 지원하지 않는다.

In [ ]:
import json
from pathlib import Path
import sys
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/security_audit/ontology.py").exists())
sys.path.insert(0, str(PROJECT))
result = json.loads((PROJECT / "experiments/results/security_audit_sft.json").read_text())
print(result["run"])

## 온톨로지와 입력 단위

- **L1 공통 엔티티**: Person, Position, Unit, Organization, Facility, Credential, ClassifiedDocument, StorageMedium, InformationSystem, Regulation, Inspection의 11종이다. 계급이 이름과 같은 칸이면 Person에 포함하며 자격 증명 값은 `****`로 가린다.
- **L2 지속 관계**: 대장은 BELONGS_TO, HOLDS_POSITION, POSITION_OF, HOLDS, RESPONSIBLE_FOR, LOCATED_AT를 추출한다. 서술 문서는 ACCESSED, CARRIED_OUT, STORED_ON, INSPECTED 같은 행위 관계도 허용한다.
- **L3 기록 레코드**: 대장 10종과 서술 사건·발견·조치 3종이다. 반출자·확인자 같은 행위 역할은 대장에서 레코드 필드로 표현한다. 직접 행위 관계로 중복 채점하지 않는다.
- **L4 기관별 표현**: 열 이름·순서·묶음 칸·날짜 형식을 데이터 변형으로 다룬다. 대장 종류는 제목으로 이미 알려진 조건이다.

관리대장 입력 하나는 **대장 공통 정보 + 평탄화한 머리글 + 행 하나**다. 서술 문서는 문단 단위로 입력한다. 구조화 레코드 필드는 해당 기록 단위를 사용한다. 실제 여러 행 사이의 상호참조·머리글 없는 입력은 측정하지 않는다.

In [ ]:
from config.security_audit.ontology import load_ontology
from config.security_audit.storage import load_docs
ontology = load_ontology()
docs = load_docs(PROJECT / "experiments/data/security_audit")
ledger = next(d for d in docs if d.get("doc_kind") == "ledger" and d["split"] == "eval")
from config.security_audit.annotation import parse_marked
print(ledger["doc_type"], ledger.get("variant_id"))
print(parse_marked(ledger["paragraphs"][0])[0])

## 데이터와 학습

시나리오 단위로 분리하여 같은 사건이 train/dev/eval에 겹치지 않게 했다. 기존 학습용 45개 중 7개를 dev로 분리했다. 관리대장은 공개 서식의 열 구성을 따라 코드로 렌더링한 합성 자료이고, 서술 자료도 가상 문서다.

| 분할 | 시나리오 | 문서 | 문단/대장 행 | 레코드 |
|---|---|---|---|---|
| train | 38 | 189 | 1216 | 970 |
| dev | 7 | 34 | 216 | 169 |
| eval | 15 | 72 | 480 | 359 |

학습은 2 epoch·배치 8·seed 42, CUDA BF16이며 평가에서는 CUDA FP32와 inference mode를 사용했다. 학습 파라미터는 2,715,664개(약 0.94%)다. 총 546 step, 233.9초가 걸렸다. 라이브러리의 학습 DataLoader는 마지막 불완전 배치를 제외하므로 2,186개 예시를 모두 매 epoch 사용한다고 해석하면 안 된다.

학습 변환 시 엔티티 경계 불일치 0개, 관계 29개, 레코드 값 14개를 정렬 사유로 제외했다. 평가 정답은 이 사유로 걸러내지 않았다. 중복 표면형 관계 단말의 발견 횟수는 741회이며 위치가 아닌 문자열로 관계를 학습하는 한계가 있다.

## 채점 기준

엔티티는 문자 시작·끝·타입의 정확 일치, 관계는 `(head 표면형, 관계, tail 표면형)`의 정확 일치로 micro F1을 계산한다. 레코드는 일반 필드의 값 집합 정확 일치로 micro F1을 계산하며 선택형 필드는 정확도로 별도 집계한다. 레코드가 여러 개 반환되면 첫 번째를 사용하는 구현이므로 한 입력에 한 기록을 주는 조건으로 해석한다.

본 이름/처음 보는 이름은 사전 소속이 아니라 실제 train에 등장한 `(표면형, 타입)` 기준이며 인명 외 모든 엔티티를 포함한다. 열 변형의 seen/unseen은 평가 전용 변형 ID 기준이다. 이름과 변형은 각각 별도로 집계하며 두 기준의 교차 조합을 분석한 결과는 아니다. 표본이 없는 선택형 그룹은 N/A다.

## dev와 최종 eval

| 조건 | 분할 | 엔티티 F1 | 관계 F1 | 레코드 필드 F1 | 선택형 정확도 |
|---|---|---|---|---|---|
| zero_shot | dev | 0.528 | 0.042 | 0.629 | 0.667 |
| zero_shot | eval | 0.444 | 0.050 | 0.594 | 0.614 |
| adapter | dev | 0.990 | 0.724 | 0.992 | 0.485 |
| adapter | eval | 0.979 | 0.772 | 0.965 | 0.493 |

## 엔티티 F1: 문서 종류·이름·열 변형

| 구분 | 그룹 | zero-shot | LoRA |
|---|---|---|---|
| 표면형 | seen | 0.613 | 0.997 |
| 표면형 | unseen | 0.391 | 0.970 |
| 열 변형(대장) | seen | 0.432 | 0.988 |
| 열 변형(대장) | unseen | 0.407 | 0.982 |
| 문서 | ledger | 0.420 | 0.985 |
| 문서 | narrative | 0.555 | 0.955 |

평가 정답 중 본 표면형은 565개, 처음 보는 표면형은 1152개다. 예측도 그 표면형의 train 등장 여부로 분류하므로 각 그룹의 precision에도 오탐이 포함된다.

### 엔티티 타입별

| 타입 | eval 정답 수 | zero-shot F1 | LoRA F1 |
|---|---|---|---|
| Person | 658 | 0.655 | 0.983 |
| Position | 35 | 0.233 | 1.000 |
| Unit | 594 | 0.329 | 0.999 |
| Organization | 0 | N/A | N/A |
| Facility | 140 | 0.800 | 0.932 |
| Credential | 51 | 0.000 | 1.000 |
| ClassifiedDocument | 99 | 0.293 | 0.985 |
| StorageMedium | 68 | 0.413 | 0.864 |
| InformationSystem | 27 | 0.482 | 0.862 |
| Regulation | 30 | 0.516 | 1.000 |
| Inspection | 15 | 0.341 | 1.000 |

Organization은 eval 정답이 없어 타입별 F1을 N/A로 표시한다. 원본 채점 JSON은 0으로 반환하며, 전체 micro F1에서는 해당 타입 오탐이 있다면 그대로 반영된다.


## 관계 F1

| 문서 | 정답 수 | zero-shot | LoRA |
|---|---|---|---|
| ledger | 447 | 0.000 | 0.782 |
| narrative | 156 | 0.171 | 0.743 |

대장에서는 지속 관계만, 서술에서는 허용된 행위 관계도 평가한다. 관계 평가는 정답 엔티티를 주입하지 않는 JointIE이므로 엔티티 누락·타입 오류의 영향이 포함된다.

| 관계 | 학습 수 | eval 정답 | zero-shot | LoRA |
|---|---|---|---|---|
| ACCESSED | 65 | 38 | 0.048 | 0.719 |
| BELONGS_TO | 644 | 251 | 0.000 | 0.825 |
| CARRIED_OUT | 12 | 4 | 0.000 | 0.174 |
| HOLDS | 98 | 51 | 0.000 | 1.000 |
| HOLDS_POSITION | 73 | 35 | 0.418 | 0.944 |
| INSPECTED | 38 | 17 | 0.000 | 0.938 |
| LOCATED_AT | 135 | 28 | 0.069 | 0.071 |
| POSITION_OF | 47 | 20 | 0.000 | 0.930 |
| RESPONSIBLE_FOR | 501 | 159 | 0.000 | 0.717 |

## 레코드 필드 F1

| 레코드 | 정답 필드 값 수 | zero-shot | LoRA |
|---|---|---|---|
| action | 30 | 0.915 | 1.000 |
| clearance | 284 | 0.567 | 0.953 |
| custodian | 184 | 0.560 | 0.992 |
| daily_check | 143 | 0.892 | 0.979 |
| document_entry | 123 | 0.462 | 0.922 |
| entry | 316 | 0.595 | 0.986 |
| finding | 30 | 0.281 | 0.967 |
| incident | 70 | 0.759 | 0.932 |
| loan | 223 | 0.474 | 0.996 |
| medium_entry | 315 | 0.628 | 0.968 |
| medium_transfer | 117 | 0.504 | 0.866 |
| reading | 201 | 0.693 | 0.966 |
| system_entry | 129 | 0.436 | 0.968 |

대장 10종과 서술 incident/finding/action을 구분한다. 선택형 값은 위 정답 필드 수와 F1에 포함되지 않는다.

| 대장 열 변형 | 정답 필드 값 수 | zero-shot | LoRA |
|---|---|---|---|
| seen | 1074 | 0.554 | 0.969 |
| unseen | 961 | 0.627 | 0.961 |

## 선택형 정확도: 정답 단어 유무

정답 선택지 문자열이 입력 본문에 그대로 있는 경우(literal)와 없는 경우(absent)를 구분한다. 본문에 다른 표현으로 의미가 있어도 정확한 문자열이 없으면 absent다.

| 필드 | literal 표본 | zero-shot | LoRA | absent 표본 | zero-shot | LoRA |
|---|---|---|---|---|---|---|
| action.조치 유형 | 13 | 1.000 | 1.000 | 2 | 1.000 | 1.000 |
| clearance.인가등급 | 48 | 0.292 | 0.000 | 0 | N/A | N/A |
| custodian.인가등급 | 40 | 0.325 | 0.000 | 0 | N/A | N/A |
| custodian.정부 구분 | 40 | 1.000 | 1.000 | 0 | N/A | N/A |
| daily_check.점검결과 | 30 | 0.733 | 1.000 | 0 | N/A | N/A |
| document_entry.비밀등급 | 13 | 0.615 | 0.000 | 0 | N/A | N/A |
| document_entry.처리방법 | 13 | 0.692 | 1.000 | 0 | N/A | N/A |
| finding.영역 | 8 | 1.000 | 1.000 | 7 | 0.429 | 0.143 |
| incident.유형 | 8 | 1.000 | 1.000 | 7 | 0.714 | 0.857 |
| loan.비밀등급 | 35 | 0.371 | 0.000 | 0 | N/A | N/A |
| medium_entry.등급 | 40 | 1.000 | 1.000 | 0 | N/A | N/A |
| medium_transfer.구분 | 14 | 0.357 | 1.000 | 0 | N/A | N/A |
| reading.인가등급 | 37 | 0.405 | 0.000 | 0 | N/A | N/A |

**비밀등급 선택형 실패:** clearance·custodian·reading의 인가등급과 document_entry·loan의 비밀등급은 총 **173건 모두 저장된 예측에서 해당 키가 없었다**. 정답과 선택지는 `Ⅰ급/Ⅱ급/Ⅲ급`이며, 단순한 오답 등급 선택이 아니라 빈 값/누락으로 기록된 실패다. 반면 medium_entry의 `비밀용/일반용`은 정상적으로 선택했다. grade 누락의 원인은 아직 규명하지 않았고, 채점 기준이나 threshold를 바꿔 점수를 보정하지 않았다.

본문에 정답이 없는 그룹의 LoRA 정확도는 action 2/2, finding 1/7, incident 6/7이다. 표본이 매우 적고 필드별 차이가 크므로 absent 전체를 잘 추론한다고 일반화하지 않는다.


## 추론 예시: 정답과 모델 출력

관리대장과 서술 문서의 앞 4개 예시를 저장했다. 이는 오류 유형 전체를 대표하는 표본이 아니며, 출력 관계는 추출된 사실일 뿐 이상 판정 결과가 아니다. 아래를 통해 엔티티·관계 간선과 기록 필드를 함께 확인한다.

In [ ]:
for kind, examples in result["examples"].items():
    print("\nDOCUMENT KIND:", kind)
    for example in examples:
        print(example["doc_id"], example["variant"], example["text"])
        print("GOLD:", example["gold"])
        print("PREDICTED:", example["predicted"])

## 시간과 자원

| 조건 | 분할 | 엔티티 | 관계 | 레코드 | 평가 합계 |
|---|---|---|---|---|---|
| zero_shot | dev | 5.4초 | 20.1초 | 2.9초 | 28.4초 |
| zero_shot | eval | 10.5초 | 41.1초 | 6.2초 | 57.9초 |
| adapter | dev | 6.4초 | 24.1초 | 3.6초 | 34.2초 |
| adapter | eval | 13.5초 | 48.9초 | 7.6초 | 69.9초 |

본 실행 전체 경과는 447.2초(모델 로딩·학습·평가·집계 포함), 학습은 233.9초다. 평가 시간은 항목별 추론 시간을 합산한 값이며 모델 로딩과 전체 채점 시간은 제외한다. 자원 상세는 run의 `resources.jsonl`에 기록했고 기존 vLLM 작업은 종료하지 않았다. 속도는 각 조건 1회 측정이므로 공유 자원의 부하에 영향을 받는다.

15초 간격 표본에서 최소 가용 메모리는 **14.9GiB**, 최대 CUDA 예약 메모리는 **16.6GiB**, 최대 CUDA 할당 메모리는 **7.0GiB**였다. 이는 표본 관측값으로 순간 최대치를 보장하지 않는다. 기존 vLLM은 유지했고 자원 중단 기준에 걸리지 않았다.

스모크는 약84초에 완료했다. 저장된 예측의 항목 수·고유 ID·순서를 확인하고, 네 조건/분할의 점수를 저장 예측으로 다시 계산해 최종 결과와 모두 일치함을 검증했다. 단위 테스트는 68개 통과했다.


## v1 시험과 비교

v1의 최종 eval은 엔티티 F1 0.945, 관계 F1 0.656, 레코드 필드 F1 0.842였다. 당시 평가에서 사람 외 이름의 상당수가 학습과 겹쳤고 문서가 서술형에 치우쳤다. v2는 표 형태 대장·새 이름·새 열 변형을 포함하며 관계/역할의 표현도 바뀌었다. **동일 데이터 비교가 아니므로 점수 증감을 모델 개선·퇴보로 해석할 수 없다.** v1 시험 데이터와 수치는 `9f18cde` 및 보존된 pilot 결과를 참조한다.

## 결론과 한계

**현재 판단:** 엔티티와 일반 필드 추출은 목표를 넘었지만, 관계는 목표에 미달하고 등급 선택형 누락이 남아 있다. 따라서 통합 어댑터의 실험 전체를 성공 또는 실문서 적용 준비 완료로 결론 내리지 않는다.

이번 최종 eval에서 엔티티 F1은 **0.444 → 0.979**, 관계 F1은 **0.050 → 0.772**, 레코드 필드 F1은 **0.594 → 0.965**였다. 전체 점수만으로 판단하지 않고 위의 처음 보는 표면형·열 변형 및 대장별·선택형 그룹의 점수와 표본 수를 함께 봐야 한다.

- 단일 seed·단일 실행이며 신뢰구간과 반복 실험은 없다. dev/eval을 이용한 하이퍼파라미터 재탐색은 하지 않았다.
- 데이터는 합성 가상 문서이며 실제 감사 문서·OCR 오류·다중 행 상호참조·새 대장 종류로 일반화되는지는 검증하지 않았다.
- 입력 단위와 스키마를 알고 있는 조건이며 기록 역할 필드와 지속 관계를 구분했다. 이 점수는 이상 판정 정확도가 아니다.
- 관계는 문자열 단말을 사용하고 같은 이름의 여러 등장 위치를 구분하지 않는다. 학습에서 정렬 실패 관계·값을 제외한 영향과 정답 누락·중복 가능성은 별도 오류 분석이 필요하다.
- 선택형 absent가 낮으면 단순 추출을 넘어선 추론 문제로 보고, 표본 불균형·라벨 설명·학습 사례를 먼저 검토한다. 목표 미달을 감추기 위해 eval을 반복 튜닝에 사용하지 않는다.
- 어댑터는 이 실험에서 병합하지 않았다. 저장된 어댑터와 Kiwi 분할기를 함께 사용해야 한다.

**다음 검토 우선순위:** 173건의 등급 선택형 누락 원인(학습 target과 선택지 토큰 정렬, 디코딩 경로)을 먼저 분석하고, 관계 오류를 대장/서술과 관계 종류별로 살펴본다. 이번 실행의 결과는 그대로 보존하며, 수정 후 재학습·재평가는 별도 실험으로 구분한다.
